# Semana 11 - Estrategia ML y diagnóstico

## Objetivo
Construir una evaluación completa: baseline, pipeline, CV, learning curve y error analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Dataset local
Usamos `load_breast_cancer`, disponible sin internet.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X.shape, y.value_counts(normalize=True).sort_index()

## Funciones de evaluación

In [ ]:
def make_splits(X, y, test_size=0.2, dev_size=0.25, random_state=42):
    X_train_dev, X_test, y_train_dev, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )
    X_train, X_dev, y_train, y_dev = train_test_split(
        X_train_dev,
        y_train_dev,
        test_size=dev_size,
        stratify=y_train_dev,
        random_state=random_state,
    )
    return X_train, X_dev, X_test, y_train, y_dev, y_test


def metric_report(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }


def build_baseline(strategy="most_frequent"):
    return DummyClassifier(strategy=strategy)


def build_pipeline(C=1.0, random_state=42):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=C,
                    max_iter=5000,
                    solver="liblinear",
                    random_state=random_state,
                ),
            ),
        ]
    )


def cross_validate_model(model, X, y, cv=5, scoring="f1"):
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
    return {"mean": float(np.mean(scores)), "std": float(np.std(scores)), "scores": scores}


def learning_curve_summary(model, X, y, train_sizes=None, cv=5, scoring="f1"):
    if train_sizes is None:
        train_sizes = np.array([0.3, 0.6, 1.0])
    sizes, train_scores, validation_scores = learning_curve(
        model,
        X,
        y,
        train_sizes=train_sizes,
        cv=cv,
        scoring=scoring,
    )
    return {
        "train_sizes": sizes,
        "train_scores_mean": train_scores.mean(axis=1),
        "validation_scores_mean": validation_scores.mean(axis=1),
    }


def slice_error_report(X, y_true, y_pred, feature, threshold):
    X_df = pd.DataFrame(X).copy()
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = X_df[feature].to_numpy() >= threshold
    rows = []
    for label, group_mask in [(f"{feature} < {threshold:.3g}", ~mask), (f"{feature} >= {threshold:.3g}", mask)]:
        yt = y_true[group_mask]
        yp = y_pred[group_mask]
        rows.append(
            {
                "slice": label,
                "n": int(group_mask.sum()),
                "f1": float(f1_score(yt, yp, zero_division=0)),
                "errors": int(np.sum(yt != yp)),
            }
        )
    return pd.DataFrame(rows)


def choose_best_model(results, metric="f1"):
    best_name = max(results, key=lambda name: results[name]["mean"])
    return best_name, float(results[best_name]["mean"])

## Particiones y baseline

In [ ]:
X_train, X_dev, X_test, y_train, y_dev, y_test = make_splits(X, y, random_state=42)
baseline = build_baseline()
baseline.fit(X_train, y_train)
metric_report(y_dev, baseline.predict(X_dev))

## Pipeline y cross-validation

In [ ]:
pipe = build_pipeline(C=1.0, random_state=42)
cv_summary = cross_validate_model(pipe, X_train, y_train, cv=5, scoring="f1")
cv_summary

## Validación y error analysis

In [ ]:
pipe.fit(X_train, y_train)
y_dev_pred = pipe.predict(X_dev)
metric_report(y_dev, y_dev_pred)

In [ ]:
threshold = float(X_dev["mean radius"].median())
slice_error_report(X_dev, y_dev, y_dev_pred, feature="mean radius", threshold=threshold)

## Learning curve

In [ ]:
learning_curve_summary(build_pipeline(C=1.0), X_train, y_train, train_sizes=[0.4, 0.7, 1.0], cv=3)

## Cierre
Escriba una recomendación en forma: evidencia -> diagnóstico -> acción.